# DC Channel Analysis: Event-Level Timing Extraction

**Purpose:** Validate that we can extract precise onset times from the DC audio channel in EDF files.

**Goals:**
1. Load CON008 EDF and inspect channel structure
2. Identify and extract DC audio channel
3. Detect beep/stimulus onsets using peak detection
4. Compare detected times with CSV timestamps
5. Measure alignment precision (target: ±50ms)

**Context:**
- CON010 uses newer stimulus software with event-level timing embedded in CSV
- Older experiments (CON008, CON009) only have trial-level timing
- DC audio channel can recover event-level timing for older data

---

## 1. Setup & Imports

In [ ]:
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from pathlib import Path
import ast
import warnings
warnings.filterwarnings('ignore')

# Set paths
DATA_ROOT = Path('../..').resolve() / 'Data' / 'extracted' / 'EEG'
EDF_DIR = DATA_ROOT / 'edf'

print(f"Data root: {DATA_ROOT}")
print(f"EDF directory: {EDF_DIR}")
print(f"\nAvailable EDF files:")
for f in EDF_DIR.glob('*.EDF'):
    print(f"  - {f.name} ({f.stat().st_size / 1e6:.1f} MB)")

## 2. Load CON008 EDF & Inspect Channels

In [ ]:
# Load CON008 clipped EDF (smaller file for faster loading)
edf_path = EDF_DIR / 'CON008_clipped.EDF'
print(f"Loading: {edf_path}")

raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)

print(f"\n=== EDF Info ===")
print(f"Sampling frequency: {raw.info['sfreq']} Hz")
print(f"Duration: {raw.times[-1]:.1f} seconds ({raw.times[-1]/60:.1f} minutes)")
print(f"Number of channels: {len(raw.ch_names)}")
print(f"Recording start: {raw.info['meas_date']}")

In [ ]:
# List all channels and identify potential DC/audio channels
print("=== All Channels ===")
for i, ch in enumerate(raw.ch_names):
    ch_type = raw.get_channel_types()[i]
    print(f"  {i:2d}. {ch:15s} (type: {ch_type})")

# Look for DC, AUX, or audio-related channels
potential_dc = [ch for ch in raw.ch_names if any(x in ch.upper() for x in ['DC', 'AUX', 'AUDIO', 'DIG'])]
print(f"\nPotential DC/Audio channels: {potential_dc}")

## 3. Extract & Visualize DC Channel

In [ ]:
# Extract DC channel (adjust name based on what we find above)
# Common names: 'DC1', 'DC', 'AUX1', 'Audio'
dc_channel_name = None

# Try to find DC channel automatically
for ch in raw.ch_names:
    if 'DC' in ch.upper() or 'AUX' in ch.upper():
        dc_channel_name = ch
        break

if dc_channel_name is None:
    print("WARNING: No DC channel found automatically.")
    print("Please inspect channels above and set dc_channel_name manually.")
    # Fallback: use last channel (often DC in EDF files)
    dc_channel_name = raw.ch_names[-1]
    print(f"Using last channel as fallback: {dc_channel_name}")

print(f"\nUsing DC channel: {dc_channel_name}")

# Extract DC signal
dc_raw = raw.copy().pick_channels([dc_channel_name])
dc_signal = dc_raw.get_data()[0]
sfreq = raw.info['sfreq']
times = raw.times

print(f"DC signal shape: {dc_signal.shape}")
print(f"DC signal stats: min={dc_signal.min():.4f}, max={dc_signal.max():.4f}, mean={dc_signal.mean():.4f}")

In [ ]:
# Plot full DC channel to see overall structure
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(times, dc_signal, linewidth=0.5)
ax.set_xlabel('Time (seconds)')
ax.set_ylabel('Amplitude')
ax.set_title(f'DC Audio Channel: {dc_channel_name} (Full Recording)')
plt.tight_layout()
plt.show()

print("Look for patterns: beeps should appear as amplitude spikes or waveform bursts")

In [ ]:
# Zoom into first 60 seconds to see beep structure
zoom_duration = 60  # seconds
zoom_samples = int(zoom_duration * sfreq)

fig, axes = plt.subplots(2, 1, figsize=(14, 6))

# Raw signal
axes[0].plot(times[:zoom_samples], dc_signal[:zoom_samples], linewidth=0.5)
axes[0].set_xlabel('Time (seconds)')
axes[0].set_ylabel('Raw Amplitude')
axes[0].set_title(f'DC Channel (First {zoom_duration}s) - Raw')

# Normalized signal (z-score)
dc_norm = (dc_signal - dc_signal.mean()) / dc_signal.std()
axes[1].plot(times[:zoom_samples], dc_norm[:zoom_samples], linewidth=0.5)
axes[1].set_xlabel('Time (seconds)')
axes[1].set_ylabel('Normalized (z-score)')
axes[1].set_title(f'DC Channel (First {zoom_duration}s) - Normalized')
axes[1].axhline(y=3, color='r', linestyle='--', alpha=0.5, label='Detection threshold (3σ)')
axes[1].axhline(y=-3, color='r', linestyle='--', alpha=0.5)
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Detect Audio Events (Peak Detection)

In [ ]:
# Peak detection parameters
# - height: minimum z-score for a peak (3 = 3 standard deviations above mean)
# - distance: minimum samples between peaks (0.8s at sampling rate)

height_threshold = 3  # z-score threshold
min_distance_sec = 0.8  # minimum time between events (seconds)
min_distance_samples = int(min_distance_sec * sfreq)

print(f"Peak detection parameters:")
print(f"  Height threshold: {height_threshold} σ")
print(f"  Min distance: {min_distance_sec}s ({min_distance_samples} samples)")

# Detect peaks in positive direction
peaks_pos, props_pos = find_peaks(dc_norm, height=height_threshold, distance=min_distance_samples)

# Also check negative peaks (audio might be inverted)
peaks_neg, props_neg = find_peaks(-dc_norm, height=height_threshold, distance=min_distance_samples)

print(f"\nPeaks detected:")
print(f"  Positive peaks: {len(peaks_pos)}")
print(f"  Negative peaks: {len(peaks_neg)}")

# Use whichever has more peaks (audio polarity varies)
if len(peaks_pos) >= len(peaks_neg):
    peaks = peaks_pos
    peak_heights = props_pos['peak_heights']
    print(f"  Using: Positive peaks")
else:
    peaks = peaks_neg
    peak_heights = props_neg['peak_heights']
    print(f"  Using: Negative peaks (inverted audio)")

In [ ]:
# Convert peak samples to timestamps
peak_times_edf = peaks / sfreq  # Time in EDF (seconds from recording start)

# Convert to Unix timestamps
edf_start_unix = raw.info['meas_date'].timestamp()
peak_times_unix = peak_times_edf + edf_start_unix

print(f"EDF recording start (Unix): {edf_start_unix}")
print(f"\nFirst 10 detected events:")
for i, (t_edf, t_unix) in enumerate(zip(peak_times_edf[:10], peak_times_unix[:10])):
    print(f"  {i+1:2d}. EDF: {t_edf:8.2f}s | Unix: {t_unix:.3f}")

In [ ]:
# Visualize detected peaks
fig, ax = plt.subplots(figsize=(14, 5))

# Plot first 120 seconds with detected peaks
plot_duration = 120
plot_samples = int(plot_duration * sfreq)

ax.plot(times[:plot_samples], dc_norm[:plot_samples], linewidth=0.5, label='DC signal (normalized)')

# Mark detected peaks within this window
peak_mask = peaks < plot_samples
peaks_in_window = peaks[peak_mask]
ax.scatter(times[peaks_in_window], dc_norm[peaks_in_window], 
           color='red', s=50, zorder=5, label=f'Detected events ({len(peaks_in_window)} in window)')

ax.axhline(y=height_threshold, color='orange', linestyle='--', alpha=0.5, label=f'Threshold ({height_threshold}σ)')
ax.set_xlabel('Time (seconds)')
ax.set_ylabel('Normalized Amplitude')
ax.set_title(f'DC Channel with Detected Events (First {plot_duration}s)')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Total events detected in full recording: {len(peaks)}")

## 5. Load CSV & Compare Timestamps

In [ ]:
# Load CON008 stimulus results CSV
csv_path = DATA_ROOT / 'CON008_2025-08-14_stimulus_results.csv'
stim_df = pd.read_csv(csv_path)

print(f"Loaded: {csv_path.name}")
print(f"Shape: {stim_df.shape}")
print(f"\nColumns: {list(stim_df.columns)}")
print(f"\nTrial types:")
print(stim_df['trial_type'].value_counts())

In [ ]:
# Inspect first few trials
print("=== First 5 Trials ===")
for idx, row in stim_df.head().iterrows():
    print(f"\nTrial {idx}: {row['trial_type']}")
    print(f"  Start: {row['start_time']:.3f}")
    print(f"  End: {row['end_time']:.3f}")
    print(f"  Duration: {row['duration']:.2f}s")
    sentences = row['sentences']
    if isinstance(sentences, str):
        try:
            parsed = ast.literal_eval(sentences)
            print(f"  Sentences: {len(parsed)} items - {str(parsed)[:80]}...")
        except:
            print(f"  Sentences: {sentences[:80]}...")

In [ ]:
# Focus on oddball trials (have clear beep structure)
oddball_trials = stim_df[stim_df['trial_type'].str.contains('oddball', case=False, na=False)].copy()
print(f"Oddball trials: {len(oddball_trials)}")

if len(oddball_trials) > 0:
    for idx, row in oddball_trials.iterrows():
        print(f"\nOddball Trial {idx}:")
        print(f"  Start: {row['start_time']:.3f}")
        print(f"  End: {row['end_time']:.3f}")
        print(f"  Duration: {row['duration']:.2f}s")
        
        # Parse stimulus sequence
        try:
            seq = ast.literal_eval(row['sentences'])
            n_standard = seq.count('standard')
            n_rare = seq.count('rare')
            print(f"  Sequence: {len(seq)} beeps ({n_standard} standard, {n_rare} rare)")
        except:
            print(f"  Sequence: {row['sentences'][:60]}...")

## 6. Validate Alignment (±50ms Target)

In [ ]:
# For each oddball trial, find detected peaks within the trial window
# and compare count + timing

alignment_results = []

for idx, row in oddball_trials.iterrows():
    trial_start = row['start_time']
    trial_end = row['end_time']
    
    # Parse expected beep count
    try:
        seq = ast.literal_eval(row['sentences'])
        expected_beeps = len(seq)
    except:
        expected_beeps = None
    
    # Find detected peaks within trial window
    mask = (peak_times_unix >= trial_start) & (peak_times_unix <= trial_end)
    detected_in_trial = peak_times_unix[mask]
    
    # Calculate timing offset from trial start
    if len(detected_in_trial) > 0:
        first_detected = detected_in_trial[0]
        offset_ms = (first_detected - trial_start) * 1000
    else:
        offset_ms = None
    
    alignment_results.append({
        'trial_idx': idx,
        'trial_start': trial_start,
        'trial_end': trial_end,
        'expected_beeps': expected_beeps,
        'detected_beeps': len(detected_in_trial),
        'first_beep_offset_ms': offset_ms
    })
    
    print(f"Trial {idx}: Expected={expected_beeps}, Detected={len(detected_in_trial)}, Offset={offset_ms:.1f}ms" if offset_ms else f"Trial {idx}: No beeps detected")

alignment_df = pd.DataFrame(alignment_results)

In [ ]:
# Calculate precision metrics
if len(alignment_df) > 0 and alignment_df['first_beep_offset_ms'].notna().any():
    offsets = alignment_df['first_beep_offset_ms'].dropna()
    
    print("=== Alignment Precision Metrics ===")
    print(f"Trials analyzed: {len(offsets)}")
    print(f"Mean offset: {offsets.mean():.1f} ms")
    print(f"Std offset: {offsets.std():.1f} ms")
    print(f"Min offset: {offsets.min():.1f} ms")
    print(f"Max offset: {offsets.max():.1f} ms")
    print(f"\nWithin ±50ms: {(offsets.abs() <= 50).mean() * 100:.1f}%")
    print(f"Within ±100ms: {(offsets.abs() <= 100).mean() * 100:.1f}%")
else:
    print("No valid alignment data to analyze.")

## 7. Compare with CON010 Format (Event-Level Timing)

In [ ]:
# Load CON010 to see the newer format with embedded onset times
con010_path = DATA_ROOT / 'CON010_2025-10-31_stimulus_results.csv'
con010_df = pd.read_csv(con010_path)

print(f"CON010 Shape: {con010_df.shape}")
print(f"\nTrial types:")
print(con010_df['trial_type'].value_counts())

print("\n=== CON010 Format Example ===")
for idx, row in con010_df.head(3).iterrows():
    print(f"\nTrial {idx}: {row['trial_type']}")
    print(f"  Start: {row['start_time']:.3f}")
    print(f"  Duration: {row['duration']:.2f}s")
    
    # Parse sentences (should be list of dicts with onset_time)
    try:
        events = ast.literal_eval(row['sentences'])
        print(f"  Events ({len(events)}):")
        for e in events[:3]:
            print(f"    - {e['event']}: {e['onset_time']:.3f}")
        if len(events) > 3:
            print(f"    ... and {len(events) - 3} more")
    except Exception as ex:
        print(f"  Sentences: {row['sentences'][:80]}...")

In [ ]:
# Side-by-side format comparison
print("=" * 80)
print("FORMAT COMPARISON: CON008 (old) vs CON010 (new)")
print("=" * 80)

print("\n--- CON008 (Old Format) ---")
print("sentences column contains: Integer indices or string labels")
print("Example (language): [10, 29, 19, 25, ...] → indices into langX.wav files")
print("Example (oddball): ['standard', 'standard', 'rare', ...] → stimulus labels")
print("Timing: Only trial-level (start_time, end_time)")
print("\nTo get event-level timing: Must extract from DC audio channel")

print("\n--- CON010 (New Format) ---")
print("sentences column contains: List of dictionaries with event + onset_time")
print("Example: [{'event': 'control_voice', 'onset_time': 1761936278.05}, ...]")
print("Timing: Event-level (onset_time for each event)")
print("\nNo DC extraction needed: Timing already embedded in CSV")

## 8. Findings & Recommendations

In [ ]:
print("=" * 80)
print("ANALYSIS SUMMARY")
print("=" * 80)

print(f"\n1. DC CHANNEL IDENTIFICATION")
print(f"   Channel found: {dc_channel_name}")
print(f"   Sampling rate: {sfreq} Hz")

print(f"\n2. EVENT DETECTION")
print(f"   Total events detected: {len(peaks)}")
print(f"   Detection method: scipy.signal.find_peaks (height={height_threshold}σ, distance={min_distance_sec}s)")

print(f"\n3. ALIGNMENT PRECISION")
if len(alignment_df) > 0 and alignment_df['first_beep_offset_ms'].notna().any():
    offsets = alignment_df['first_beep_offset_ms'].dropna()
    print(f"   Mean offset: {offsets.mean():.1f} ms")
    print(f"   Within ±50ms target: {(offsets.abs() <= 50).mean() * 100:.1f}%")
else:
    print("   No oddball trials available for precision measurement")

print(f"\n4. SCHEMA RECOMMENDATION")
print(f"   Adopt relational schema:")
print(f"   - trials.parquet: Trial-level metadata (patient_id, date, trial_type, start_time, end_time)")
print(f"   - events.parquet: Event-level timing (trial_id, event_type, onset_time, stimulus_id)")
print(f"   ")
print(f"   Benefits:")
print(f"   - Unified format for both old (DC-extracted) and new (CON010) data")
print(f"   - Supports downstream oddball (±50ms) and language (word-level) analysis")
print(f"   - Extensible for future event types")

## Next Steps

Based on this analysis:

1. **If DC extraction achieves ±50ms precision:**
   - Proceed with relational schema implementation
   - Create `src/event_extraction.py` to automate DC channel processing
   - Generate `events.parquet` for all patients

2. **If precision is insufficient:**
   - Investigate alternative detection methods (cross-correlation, envelope detection)
   - Consider using audio stimulus files for template matching
   - May need manual validation for critical trials

3. **Schema Migration:**
   - Convert unified CSV to `trials.parquet`
   - Parse CON010 dict format directly into `events.parquet`
   - Extract events from DC channel for older files
   - Document schema in `docs/SCHEMA.md`